<a href="https://colab.research.google.com/github/AzaharaAED/Proyecto_ecosistema/blob/main/cuestionario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import pandas as pd

# 1. CONFIGURACIÓN

def obtener_rutas_persistencia():
    return {
        "usuarios_csv": "usuarios.csv",
        "perfiles_json": "perfiles_usuario.json"
    }

# 2. DATOS INICIALES

def crear_usuarios_iniciales():
    return pd.DataFrame([
        ["u01", "Valero", 36, "M", 177],
        ["u02", "1hijo", 7, "M", 130],
        ["u03", "2hijo", 5, "M", 108],
        ["u04", "3hijo", 3, "F", 96],
    ], columns=["user_id", "nombre", "edad", "sexo", "altura_cm"])


def crear_perfiles_iniciales():
    return {
        "u01": {
            "nombre": "Valero",
            "edad": 36,
            "sexo": "M",
            "altura_cm": 177,
            "alergias": [],
            "preferencias": ["pollo", "arroz", "pasta", "verduras a la plancha", "huevos"],
            "alimentos_no_gustan": ["brócoli", "coliflor"],
            "objetivo": "mantenerse",
            "tipo_dieta": "equilibrada"
        },
        "u02": {
            "nombre": "1hijo",
            "edad": 7,
            "sexo": "M",
            "altura_cm": 130,
            "alergias": [],
            "preferencias": ["pasta", "pollo", "arroz", "plátano", "yogur"],
            "alimentos_no_gustan": ["espinacas", "pescado azul"],
            "objetivo": "crecer sano",
            "tipo_dieta": "infantil equilibrada"
        },
        "u03": {
            "nombre": "2hijo",
            "edad": 5,
            "sexo": "M",
            "altura_cm": 108,
            "alergias": [],
            "preferencias": ["arroz", "huevo", "pollo", "manzana", "galletas"],
            "alimentos_no_gustan": ["verduras verdes", "lentejas"],
            "objetivo": "crecer sano",
            "tipo_dieta": "infantil equilibrada"
        },
        "u04": {
            "nombre": "3hijo",
            "edad": 3,
            "sexo": "F",
            "altura_cm": 96,
            "alergias": [],
            "preferencias": ["purés", "plátano", "yogur", "pollo", "arroz"],
            "alimentos_no_gustan": ["verduras amargas", "pescado"],
            "objetivo": "crecer sano",
            "tipo_dieta": "infantil equilibrada"
        }
    }

# 3. CARGA Y GUARDADO

def cargar_o_crear_usuarios(usuarios_csv="usuarios.csv"):
    if os.path.exists(usuarios_csv):
        usuarios_df = pd.read_csv(usuarios_csv)
    else:
        usuarios_df = crear_usuarios_iniciales()
        usuarios_df.to_csv(usuarios_csv, index=False)

    return usuarios_df


def cargar_o_crear_perfiles(perfiles_json="perfiles_usuario.json"):
    if os.path.exists(perfiles_json):
        with open(perfiles_json, "r", encoding="utf-8") as f:
            perfiles = json.load(f)
    else:
        perfiles = crear_perfiles_iniciales()
        with open(perfiles_json, "w", encoding="utf-8") as f:
            json.dump(perfiles, f, ensure_ascii=False, indent=4)

    return perfiles


def guardar_usuarios(usuarios_df, usuarios_csv="usuarios.csv"):
    usuarios_df.to_csv(usuarios_csv, index=False)


def guardar_perfiles(perfiles_dict, perfiles_json="perfiles_usuario.json"):
    with open(perfiles_json, "w", encoding="utf-8") as f:
        json.dump(perfiles_dict, f, ensure_ascii=False, indent=4)


# 4. FUNCIONES AUXILIARES

def limpiar_lista_texto(texto):
    texto = str(texto).strip()
    if texto == "":
        return []

    return [x.strip().lower() for x in texto.split(",") if x.strip()]


def generar_nuevo_user_id(usuarios_df):
    if len(usuarios_df) == 0:
        return "u01"

    numeros = []
    for uid in usuarios_df["user_id"]:
        uid = str(uid)
        if uid.startswith("u") and uid[1:].isdigit():
            numeros.append(int(uid[1:]))

    if len(numeros) == 0:
        return "u01"

    nuevo_num = max(numeros) + 1
    return f"u{nuevo_num:02d}"


def normalizar_entero(valor):
    try:
        if valor is None or str(valor).strip() == "":
            return None
        return int(valor)
    except (TypeError, ValueError):
        return None


def obtener_usuario_por_user_id(user_id, usuarios_df):
    coincidencias = usuarios_df[usuarios_df["user_id"] == user_id]

    if coincidencias.empty:
        return None

    fila = coincidencias.iloc[0]
    return {
        "user_id": fila["user_id"],
        "nombre": fila["nombre"],
        "edad": normalizar_entero(fila["edad"]),
        "sexo": fila["sexo"],
        "altura_cm": normalizar_entero(fila["altura_cm"])
    }


def obtener_user_id_por_nombre(nombre, usuarios_df):
    nombre = str(nombre).strip().lower()

    coincidencias = usuarios_df[
        usuarios_df["nombre"].astype(str).str.strip().str.lower() == nombre
    ]

    if coincidencias.empty:
        return None

    return coincidencias.iloc[0]["user_id"]


# 5. CREAR / ACTUALIZAR USUARIOS Y PERFILES

def registrar_nuevo_usuario(usuarios_df, nombre, edad, sexo, altura_cm):
    nuevo_user_id = generar_nuevo_user_id(usuarios_df)

    nuevo_usuario = pd.DataFrame([[
        nuevo_user_id,
        str(nombre).strip(),
        normalizar_entero(edad),
        str(sexo).strip(),
        normalizar_entero(altura_cm)
    ]], columns=["user_id", "nombre", "edad", "sexo", "altura_cm"])

    usuarios_df_actualizado = pd.concat([usuarios_df, nuevo_usuario], ignore_index=True)

    return {
        "user_id": nuevo_user_id,
        "nombre": str(nombre).strip(),
        "edad": normalizar_entero(edad),
        "sexo": str(sexo).strip(),
        "altura_cm": normalizar_entero(altura_cm),
        "usuarios_df_actualizado": usuarios_df_actualizado
    }


def construir_respuestas_cuestionario(
    alergias="",
    preferencias="",
    alimentos_no_gustan="",
    objetivo="",
    tipo_dieta=""
):
    return {
        "alergias": limpiar_lista_texto(alergias),
        "preferencias": limpiar_lista_texto(preferencias),
        "alimentos_no_gustan": limpiar_lista_texto(alimentos_no_gustan),
        "objetivo": str(objetivo).strip().lower(),
        "tipo_dieta": str(tipo_dieta).strip().lower()
    }


def crear_o_actualizar_perfil_usuario(
    user_id,
    usuarios_df,
    perfiles_dict,
    respuestas_cuestionario=None
):
    usuario = obtener_usuario_por_user_id(user_id, usuarios_df)

    if usuario is None:
        return None, perfiles_dict

    perfil_existente = perfiles_dict.get(user_id)

    if perfil_existente is not None and respuestas_cuestionario is None:
        return {
            "user_id": user_id,
            "perfil_usuario": perfil_existente
        }, perfiles_dict

    if respuestas_cuestionario is None:
        respuestas_cuestionario = construir_respuestas_cuestionario()

    perfiles_dict[user_id] = {
        "nombre": usuario["nombre"],
        "edad": usuario["edad"],
        "sexo": usuario["sexo"],
        "altura_cm": usuario["altura_cm"],
        "alergias": respuestas_cuestionario["alergias"],
        "preferencias": respuestas_cuestionario["preferencias"],
        "alimentos_no_gustan": respuestas_cuestionario["alimentos_no_gustan"],
        "objetivo": respuestas_cuestionario["objetivo"],
        "tipo_dieta": respuestas_cuestionario["tipo_dieta"]
    }

    return {
        "user_id": user_id,
        "perfil_usuario": perfiles_dict[user_id]
    }, perfiles_dict


# 6. FUNCIONES INTERACTIVAS

def seleccionar_o_registrar_usuario_interactivo(usuarios_df):
    print("¿Quién eres?")
    for i, row in usuarios_df.iterrows():
        print(f"{i+1}. {row['nombre']}")
    print(f"{len(usuarios_df)+1}. Otro")

    opcion = input("Selecciona una opción: ").strip()

    mapa = {str(i + 1): usuarios_df.iloc[i]["user_id"] for i in range(len(usuarios_df))}
    opcion_otro = str(len(usuarios_df) + 1)

    if opcion in mapa:
        user_id = mapa[opcion]
        usuario = obtener_usuario_por_user_id(user_id, usuarios_df)
        usuario["es_nuevo"] = False
        usuario["usuarios_df_actualizado"] = usuarios_df
        return usuario

    if opcion == opcion_otro:
        print("\nVamos a registrar un nuevo usuario.")
        nombre = input("Nombre: ").strip()
        edad = input("Edad: ").strip()
        sexo = input("Sexo (F/M/Otro): ").strip()
        altura_cm = input("Altura en cm: ").strip()

        nuevo_usuario = registrar_nuevo_usuario(
            usuarios_df=usuarios_df,
            nombre=nombre,
            edad=edad,
            sexo=sexo,
            altura_cm=altura_cm
        )

        nuevo_usuario["es_nuevo"] = True
        return nuevo_usuario

    print("Opción no válida.")
    return None


def hacer_cuestionario_inicial_interactivo(nombre):
    print(f"\nVamos a completar el cuestionario de {nombre}.")
    print("Si hay varias respuestas, sepáralas por comas.")

    alergias = input("¿Tienes alergias o intolerancias?: ").strip()
    preferencias = input("¿Qué alimentos o tipos de comida prefieres?: ").strip()
    no_gustan = input("¿Qué alimentos no te gustan?: ").strip()
    objetivo = input("¿Cuál es tu objetivo principal? (ej: perder grasa, ganar masa muscular, comer mejor): ").strip()
    tipo_dieta = input("¿Sigues algún tipo de dieta? (ej: vegetariana, vegana, omnivora, sin lactosa...): ").strip()

    return construir_respuestas_cuestionario(
        alergias=alergias,
        preferencias=preferencias,
        alimentos_no_gustan=no_gustan,
        objetivo=objetivo,
        tipo_dieta=tipo_dieta
    )


def obtener_o_crear_perfil_usuario_interactivo(
    usuarios_df,
    perfiles_dict,
    usuarios_csv="usuarios.csv",
    perfiles_json="perfiles_usuario.json"
):
    seleccion = seleccionar_o_registrar_usuario_interactivo(usuarios_df)

    if seleccion is None:
        return None, usuarios_df, perfiles_dict

    user_id = seleccion["user_id"]
    nombre = seleccion["nombre"]
    usuarios_df = seleccion["usuarios_df_actualizado"]

    necesita_cuestionario = seleccion.get("es_nuevo", False) or (user_id not in perfiles_dict)

    if necesita_cuestionario:
        respuestas = hacer_cuestionario_inicial_interactivo(nombre)

        resultado, perfiles_dict = crear_o_actualizar_perfil_usuario(
            user_id=user_id,
            usuarios_df=usuarios_df,
            perfiles_dict=perfiles_dict,
            respuestas_cuestionario=respuestas
        )

        guardar_usuarios(usuarios_df, usuarios_csv=usuarios_csv)
        guardar_perfiles(perfiles_dict, perfiles_json=perfiles_json)
    else:
        resultado, perfiles_dict = crear_o_actualizar_perfil_usuario(
            user_id=user_id,
            usuarios_df=usuarios_df,
            perfiles_dict=perfiles_dict,
            respuestas_cuestionario=None
        )

    return resultado, usuarios_df, perfiles_dict


# 7. FUNCIÓN FINAL DEL MÓDULO

def inicializar_modulo_cuestionario(
    usuarios_csv="usuarios.csv",
    perfiles_json="perfiles_usuario.json"
):
    usuarios_df = cargar_o_crear_usuarios(usuarios_csv=usuarios_csv)
    perfiles_dict = cargar_o_crear_perfiles(perfiles_json=perfiles_json)

    return {
        "usuarios_df": usuarios_df,
        "perfiles_dict": perfiles_dict,
        "usuarios_csv": usuarios_csv,
        "perfiles_json": perfiles_json
    }


def obtener_cuestionario_para_modelo_interactivo(
    usuarios_csv="usuarios.csv",
    perfiles_json="perfiles_usuario.json"
):
    estado = inicializar_modulo_cuestionario(
        usuarios_csv=usuarios_csv,
        perfiles_json=perfiles_json
    )

    resultado, usuarios_df, perfiles_dict = obtener_o_crear_perfil_usuario_interactivo(
        usuarios_df=estado["usuarios_df"],
        perfiles_dict=estado["perfiles_dict"],
        usuarios_csv=usuarios_csv,
        perfiles_json=perfiles_json
    )

    return resultado, usuarios_df, perfiles_dict


# 8. FUNCIÓN NO INTERACTIVA PARA EL MODELO GENERAL

def obtener_cuestionario_para_modelo_por_user_id(
    user_id,
    usuarios_df,
    perfiles_dict
):
    resultado, perfiles_dict = crear_o_actualizar_perfil_usuario(
        user_id=user_id,
        usuarios_df=usuarios_df,
        perfiles_dict=perfiles_dict,
        respuestas_cuestionario=None
    )

    return resultado


def registrar_usuario_y_cuestionario_para_modelo(
    usuarios_df,
    perfiles_dict,
    nombre,
    edad,
    sexo,
    altura_cm,
    alergias="",
    preferencias="",
    alimentos_no_gustan="",
    objetivo="",
    tipo_dieta="",
    usuarios_csv=None,
    perfiles_json=None
):
    nuevo_usuario = registrar_nuevo_usuario(
        usuarios_df=usuarios_df,
        nombre=nombre,
        edad=edad,
        sexo=sexo,
        altura_cm=altura_cm
    )

    usuarios_df_actualizado = nuevo_usuario["usuarios_df_actualizado"]
    user_id = nuevo_usuario["user_id"]

    respuestas = construir_respuestas_cuestionario(
        alergias=alergias,
        preferencias=preferencias,
        alimentos_no_gustan=alimentos_no_gustan,
        objetivo=objetivo,
        tipo_dieta=tipo_dieta
    )

    resultado, perfiles_dict_actualizado = crear_o_actualizar_perfil_usuario(
        user_id=user_id,
        usuarios_df=usuarios_df_actualizado,
        perfiles_dict=perfiles_dict,
        respuestas_cuestionario=respuestas
    )

    if usuarios_csv is not None:
        guardar_usuarios(usuarios_df_actualizado, usuarios_csv=usuarios_csv)

    if perfiles_json is not None:
        guardar_perfiles(perfiles_dict_actualizado, perfiles_json=perfiles_json)

    return {
        "resultado": resultado,
        "usuarios_df": usuarios_df_actualizado,
        "perfiles_dict": perfiles_dict_actualizado
    }


# 9. GUARDAR RESULTADO CONCRETO

def guardar_resultado_cuestionario(resultado, nombre_archivo="resultado_cuestionario.json"):
    with open(nombre_archivo, "w", encoding="utf-8") as f:
        json.dump(resultado, f, ensure_ascii=False, indent=4)